# 📊 Crypto Analysis — BTC + Portfolio Binance

Notebook unificado: indicadores técnicos, scoring, gráfico interactivo y portfolio.

| Sección | Contenido |
|---|---|
| 1. Datos | Descarga OHLCV desde Yahoo Finance |
| 2. Indicadores | EMAs, RSI, MACD, ADX, ATR, OBV, Bollinger, MFI, Estocástico |
| 3. Scoring | Sistema +/-20, recomendación y parámetros OCO Binance |
| 4. Gráfico | Chart interactivo Plotly con 5 paneles |
| 5. Portfolio | Balance Binance via API (keys en variables de entorno) |
| 6. Reporte HTML | Archivo HTML completo auto-contenido |
| 7. ETFs | Análisis comparativo de ETFs con mismo sistema de scoring |

> **Cambia `CHOICE`** en la celda de Configuración para seleccionar el timeframe.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import talib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import os, json, warnings
import requests, hmac, hashlib, time
from IPython.display import display, HTML
warnings.filterwarnings('ignore')
print('✅ Imports OK')

In [ ]:
# Cambia CHOICE: 1=1d  2=12h  3=6h  4=4h  5=1h  6=30m  7=15m  8=5m  9=1m
TICKER = 'BTC-USD'
CHOICE = '6'

TIMEFRAMES = {
    '1': ('1d',  '2y',  '1 DIA'),
    '2': ('1h',  '60d', '12 HORAS'),
    '3': ('1h',  '60d', '6 HORAS'),
    '4': ('1h',  '60d', '4 HORAS'),
    '5': ('1h',  '7d',  '1 HORA'),
    '6': ('30m', '7d',  '30 MIN'),
    '7': ('15m', '5d',  '15 MIN'),
    '8': ('5m',  '5d',  '5 MIN'),
    '9': ('1m',  '1d',  '1 MIN'),
}

interval, periodo, timeframe = TIMEFRAMES[CHOICE]
print(f'Timeframe: {timeframe}  ({interval} / {periodo})')

## 📥 1. Datos

Descarga OHLCV desde Yahoo Finance y convierte a arrays NumPy para TA-Lib.

In [ ]:
print(f'Descargando {TICKER} ({timeframe})...')
raw = yf.download(TICKER, period=periodo, interval=interval, progress=False)
data = raw.copy()
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.droplevel(1)
if data.empty:
    raise SystemExit('No se pudieron descargar datos')

close  = np.asarray(data['Close'].dropna().values,  dtype=float).flatten()
high   = np.asarray(data['High'].dropna().values,   dtype=float).flatten()
low    = np.asarray(data['Low'].dropna().values,    dtype=float).flatten()
volume = np.asarray(data['Volume'].dropna().values, dtype=float).flatten()
open_p = np.asarray(data['Open'].dropna().values,   dtype=float).flatten()

d0 = data.index[0].strftime('%d/%m/%Y')
d1 = data.index[-1].strftime('%d/%m/%Y %H:%M')
print(f'{len(data)} velas  |  {d0} -> {d1}')
print(f'Precio actual: ${float(close[-1]):,.2f}')
data.tail()

## 📊 2. Indicadores Técnicos

Cálculo completo con TA-Lib: EMAs, RSI, MACD, Estocástico, ADX, ATR, OBV, Bollinger, MFI.

In [ ]:
ema_4    = talib.EMA(close, timeperiod=4)
ema_20   = talib.EMA(close, timeperiod=20)
ema_25   = talib.EMA(close, timeperiod=25)
ema_50   = talib.EMA(close, timeperiod=50)
ema_200  = talib.EMA(close, timeperiod=200)
rsi      = talib.RSI(close, timeperiod=14)
rsi_5    = talib.RSI(close, timeperiod=5)
macd_v, signal_v, hist_v = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
slowk, slowd = talib.STOCH(high, low, close, fastk_period=14, slowk_period=3, slowd_period=3)
adx      = talib.ADX(high, low, close, timeperiod=14)
plus_di  = talib.PLUS_DI(high, low, close, timeperiod=14)
minus_di = talib.MINUS_DI(high, low, close, timeperiod=14)
atr      = talib.ATR(high, low, close, timeperiod=14)
obv      = talib.OBV(close, volume)
obv_ema  = talib.EMA(obv, timeperiod=20)
upper, middle, lower_bb = talib.BBANDS(close, timeperiod=20, nbdevup=2, nbdevdn=2)
mfi      = talib.MFI(high, low, close, volume, timeperiod=14)

precio  = float(close[-1])
atr_val = float(atr[-1]) if not np.isnan(atr[-1]) else 0
bb_w    = float((upper[-1]-lower_bb[-1])/middle[-1]*100) if not np.isnan(upper[-1]) else 0

df_ind = pd.DataFrame({
    'Indicador': [
        'EMA 4','EMA 20','EMA 25','EMA 50','EMA 200',
        'RSI (14)','RSI (5)','MACD','Signal','Histograma',
        'ADX','+DI','-DI','ATR','Estoc. %K','Estoc. %D',
        'MFI','BB Superior','BB Medio','BB Inferior','BB Ancho %'
    ],
    'Valor': [
        f'${ema_4[-1]:,.2f}',   f'${ema_20[-1]:,.2f}',  f'${ema_25[-1]:,.2f}',
        f'${ema_50[-1]:,.2f}',  f'${ema_200[-1]:,.2f}',
        f'{rsi[-1]:.2f}',       f'{rsi_5[-1]:.2f}',
        f'{macd_v[-1]:.4f}',   f'{signal_v[-1]:.4f}',  f'{hist_v[-1]:.4f}',
        f'{adx[-1]:.2f}',      f'{plus_di[-1]:.2f}',   f'{minus_di[-1]:.2f}',
        f'${atr[-1]:,.2f}',    f'{slowk[-1]:.2f}',      f'{slowd[-1]:.2f}',
        f'{mfi[-1]:.2f}',
        f'${upper[-1]:,.2f}',  f'${middle[-1]:,.2f}',  f'${lower_bb[-1]:,.2f}',
        f'{bb_w:.2f}%'
    ]
}).set_index('Indicador')

display(df_ind)

## 🎯 3. Scoring y Recomendación

Sistema de **+/-20 puntos** — cada indicador aporta señales. Incluye parámetros para orden OCO en Binance.

In [ ]:
puntos = 0
senales = []

# EMAs
if not any(np.isnan([ema_20[-1], ema_50[-1], ema_200[-1]])):
    if ema_20[-1] > ema_50[-1] > ema_200[-1]:
        puntos += 3; tendencia = 'ALCISTA'; senales.append('✅ EMAs alcistas (20>50>200)')
    elif ema_20[-1] < ema_50[-1] < ema_200[-1]:
        puntos -= 3; tendencia = 'BAJISTA'; senales.append('❌ EMAs bajistas (20<50<200)')
    else:
        tendencia = 'NEUTRAL'; senales.append('⚠️ EMAs sin alineacion')
else:
    tendencia = 'NEUTRAL'; senales.append('⚠️ EMAs sin datos')

if not np.isnan(ema_4[-1]):
    if precio > ema_4[-1]:  puntos += 1; senales.append(f'✅ Precio > EMA4 (${ema_4[-1]:,.2f})')
    else:                   puntos -= 1; senales.append(f'❌ Precio < EMA4 (${ema_4[-1]:,.2f})')

if not np.isnan(rsi[-1]):
    if   rsi[-1] < 30: puntos += 2; senales.append(f'✅ RSI sobreventa ({rsi[-1]:.1f})')
    elif rsi[-1] > 70: puntos -= 2; senales.append(f'❌ RSI sobrecompra ({rsi[-1]:.1f})')
    elif rsi[-1] > 50: puntos += 1; senales.append(f'✅ RSI alcista ({rsi[-1]:.1f})')
    else:              puntos -= 1; senales.append(f'❌ RSI bajista ({rsi[-1]:.1f})')

if not np.isnan(rsi_5[-1]):
    if   rsi_5[-1] < 20: puntos += 1; senales.append(f'✅ RSI5 sobreventa extrema ({rsi_5[-1]:.1f})')
    elif rsi_5[-1] > 80: puntos -= 1; senales.append(f'❌ RSI5 sobrecompra extrema ({rsi_5[-1]:.1f})')

if not any(np.isnan([macd_v[-1], signal_v[-1]])):
    if macd_v[-1] > signal_v[-1]: puntos += 2; senales.append('✅ MACD > Signal')
    else:                          puntos -= 2; senales.append('❌ MACD < Signal')
    if not any(np.isnan([hist_v[-1], hist_v[-2], hist_v[-3]])):
        if   hist_v[-1] > hist_v[-2] > hist_v[-3]: puntos += 1; senales.append('✅ Histograma MACD creciendo')
        elif hist_v[-1] < hist_v[-2] < hist_v[-3]: puntos -= 1; senales.append('❌ Histograma MACD decreciendo')

if not np.isnan(adx[-1]):
    if adx[-1] > 25:
        if plus_di[-1] > minus_di[-1]: puntos += 2; senales.append(f'✅ Tendencia fuerte alcista (ADX {adx[-1]:.1f})')
        else:                           puntos -= 2; senales.append(f'❌ Tendencia fuerte bajista (ADX {adx[-1]:.1f})')
    else: senales.append(f'⚠️ ADX debil ({adx[-1]:.1f})')

if not np.isnan(slowk[-1]):
    if slowk[-1] < 20 and slowd[-1] < 20:   puntos += 1; senales.append('✅ Estocastico sobreventa')
    elif slowk[-1] > 80 and slowd[-1] > 80: puntos -= 1; senales.append('❌ Estocastico sobrecompra')
    if not any(np.isnan([slowk[-2], slowd[-2]])):
        if   slowk[-1] > slowd[-1] and slowk[-2] < slowd[-2]: puntos += 1; senales.append('✅ Cruce alcista estocastico')
        elif slowk[-1] < slowd[-1] and slowk[-2] > slowd[-2]: puntos -= 1; senales.append('❌ Cruce bajista estocastico')

if not np.isnan(mfi[-1]):
    if   mfi[-1] < 20: puntos += 1; senales.append(f'✅ MFI sobreventa ({mfi[-1]:.1f})')
    elif mfi[-1] > 80: puntos -= 1; senales.append(f'❌ MFI sobrecompra ({mfi[-1]:.1f})')

if not any(np.isnan([obv[-1], obv_ema[-1]])):
    if obv[-1] > obv_ema[-1]: puntos += 1; senales.append('✅ OBV > EMA (presion compradora)')
    else:                      puntos -= 1; senales.append('❌ OBV < EMA (presion vendedora)')

if not np.isnan(lower_bb[-1]):
    if   precio < lower_bb[-1]: puntos += 1; senales.append('✅ Precio bajo banda Bollinger inferior')
    elif precio > upper[-1]:    puntos -= 1; senales.append('❌ Precio sobre banda Bollinger superior')
    if bb_w < 2: senales.append(f'⚠️ Bollinger comprimidas ({bb_w:.2f}%)')

# Recomendacion
if   puntos >= 8:  rec='COMPRA FUERTE'; c='#00c853'; tp=precio+atr_val*3;   act_sl=precio-atr_val*1;   sl=precio-atr_val*1.5
elif puntos >= 4:  rec='COMPRA';         c='#64dd17'; tp=precio+atr_val*2.5; act_sl=precio-atr_val*1;   sl=precio-atr_val*1.5
elif puntos >= 1:  rec='COMPRA DEBIL';   c='#ffd600'; tp=precio+atr_val*2;   act_sl=precio-atr_val*0.8; sl=precio-atr_val*1.2
elif puntos <= -8: rec='VENTA FUERTE';   c='#ff1744'; tp=precio-atr_val*3;   act_sl=precio+atr_val*1;   sl=precio+atr_val*1.5
elif puntos <= -4: rec='VENTA';          c='#ff5252'; tp=precio-atr_val*2.5; act_sl=precio+atr_val*1;   sl=precio+atr_val*1.5
elif puntos <= -1: rec='VENTA DEBIL';    c='#ff9100'; tp=precio-atr_val*2;   act_sl=precio+atr_val*0.8; sl=precio+atr_val*1.2
else:              rec='NEUTRAL';         c='#9e9e9e'; tp=act_sl=sl=None

rr = abs((tp - precio) / (precio - act_sl)) if tp and act_sl and (precio - act_sl) != 0 else 0

print(f'Puntuacion: {puntos:+d}/20  |  {rec}  |  {tendencia}')
if tp:
    print(f'TP: ${tp:,.2f}  |  Activador SL: ${act_sl:,.2f}  |  SL: ${sl:,.2f}  |  R:R 1:{rr:.2f}')

In [ ]:
rows_s = ''.join(
    '<tr><td style="padding:4px 8px;color:#888">' + str(i+1) + '</td>'
    '<td style="padding:4px 8px">' + s + '</td></tr>'
    for i, s in enumerate(senales)
)
if tp:
    oco_s = (
        '<tr><td><b>TP Limit</b></td><td style="color:#00c853;font-weight:bold">$' + f'{tp:,.2f}' + '</td></tr>'
        '<tr><td><b>Activador SL</b></td><td style="color:#ff9100;font-weight:bold">$' + f'{act_sl:,.2f}' + '</td></tr>'
        '<tr><td><b>SL Limit</b></td><td style="color:#ff1744;font-weight:bold">$' + f'{sl:,.2f}' + '</td></tr>'
        '<tr><td><b>Risk/Reward</b></td><td>1:' + f'{rr:.2f}' + '</td></tr>'
    )
else:
    oco_s = '<tr><td colspan="2" style="color:#ff9100">No operar — condiciones no favorables</td></tr>'

summary = (
    '<div style="font-family:monospace;background:#111;color:#eee;padding:20px;border-radius:10px;max-width:960px">'
    '<h2 style="text-align:center;color:' + c + ';margin:0 0 8px">' + rec + '</h2>'
    '<p style="text-align:center;margin:0 0 16px">Puntuacion <b>' + f'{puntos:+d}' + '/20</b>'
    ' &nbsp;|&nbsp; ' + tendencia + ' &nbsp;|&nbsp; Precio <b>$' + f'{precio:,.2f}' + '</b></p>'
    '<div style="display:grid;grid-template-columns:1fr 1fr;gap:16px">'
    '<div><h3 style="color:#667eea;margin:0 0 8px">Orden OCO</h3>'
    '<table style="border-collapse:collapse;width:100%">'
    '<tr><td style="padding:4px 8px"><b>Precio actual</b></td><td style="padding:4px 8px">$' + f'{precio:,.2f}' + '</td></tr>'
    + oco_s + '</table></div>'
    '<div><h3 style="color:#667eea;margin:0 0 8px">Señales (' + str(len(senales)) + ')</h3>'
    '<table style="border-collapse:collapse;width:100%;font-size:12px">' + rows_s + '</table></div>'
    '</div></div>'
)
display(HTML(summary))

## 📈 4. Gráfico Interactivo

5 paneles: Precio + EMAs + Bollinger, Volumen, RSI, MACD, Estocástico.

In [ ]:
fig = make_subplots(
    rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.04,
    subplot_titles=('Precio + EMAs + BB', 'Volumen', 'RSI', 'MACD', 'Estocastico'),
    row_heights=[0.40, 0.10, 0.15, 0.15, 0.15]
)

fig.add_trace(go.Candlestick(x=data.index, open=open_p, high=high, low=low, close=close, name='BTC'), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=ema_4,   mode='lines', name='EMA 4',   line=dict(color='#ab47bc', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=ema_20,  mode='lines', name='EMA 20',  line=dict(color='#fdd835', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=ema_25,  mode='lines', name='EMA 25',  line=dict(color='#29b6f6', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=ema_50,  mode='lines', name='EMA 50',  line=dict(color='#ef5350', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=ema_200, mode='lines', name='EMA 200', line=dict(color='#ffffff', width=1, dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=upper,    mode='lines', name='BB Sup', line=dict(color='#78909c', width=1, dash='dot')), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=lower_bb, mode='lines', name='BB Inf',
                         line=dict(color='#78909c', width=1, dash='dot'),
                         fill='tonexty', fillcolor='rgba(120,144,156,0.06)'), row=1, col=1)

fig.add_trace(go.Bar(x=data.index, y=volume/1e6, name='Vol (M)', marker_color='#4472c4'), row=2, col=1)

fig.add_trace(go.Scatter(x=data.index, y=rsi,   mode='lines', name='RSI 14', line=dict(color='#ab47bc')), row=3, col=1)
fig.add_trace(go.Scatter(x=data.index, y=rsi_5, mode='lines', name='RSI 5',  line=dict(color='#00e5ff', width=1)), row=3, col=1)
fig.add_hline(y=70, row=3, col=1, line_dash='dash', line_color='#ef5350', line_width=1)
fig.add_hline(y=30, row=3, col=1, line_dash='dash', line_color='#66bb6a', line_width=1)
fig.add_hrect(y0=30, y1=70, row=3, col=1, fillcolor='rgba(255,255,255,0.03)', line_width=0)

fig.add_trace(go.Scatter(x=data.index, y=macd_v,   mode='lines', name='MACD',   line=dict(color='#42a5f5')), row=4, col=1)
fig.add_trace(go.Scatter(x=data.index, y=signal_v, mode='lines', name='Signal', line=dict(color='#ef5350')), row=4, col=1)
hcol = ['#00c853' if h > 0 else '#ff1744' for h in hist_v]
fig.add_trace(go.Bar(x=data.index, y=hist_v, name='Hist', marker_color=hcol), row=4, col=1)

fig.add_trace(go.Scatter(x=data.index, y=slowk, mode='lines', name='%K', line=dict(color='#42a5f5')), row=5, col=1)
fig.add_trace(go.Scatter(x=data.index, y=slowd, mode='lines', name='%D', line=dict(color='#ef5350')), row=5, col=1)
fig.add_hline(y=80, row=5, col=1, line_dash='dash', line_color='#ef5350', line_width=1)
fig.add_hline(y=20, row=5, col=1, line_dash='dash', line_color='#66bb6a', line_width=1)

fig.update_layout(
    title=dict(text=f'BTC-USD | {timeframe} | {rec} | Score: {puntos:+d}/20', font=dict(size=15)),
    height=1200, showlegend=True, template='plotly_dark',
    xaxis_rangeslider_visible=False,
    legend=dict(orientation='h', y=1.01, x=0, font=dict(size=9))
)
fig.show()

## 🔸 5. Portfolio Binance

Configura las variables de entorno antes de ejecutar:
```bash
export BINANCE_API_KEY="tu_key"
export BINANCE_API_SECRET="tu_secret"
```
Sin claves: muestra solo la estructura para uso manual.

In [ ]:
class BinancePortfolio:
    BASE = 'https://api.binance.com'

    def __init__(self, api_key='', api_secret=''):
        self.api_key    = api_key
        self.api_secret = api_secret
        self.assets     = []

    def _sign(self, params):
        qs = '&'.join(str(k)+'='+str(v) for k, v in params.items())
        return hmac.new(self.api_secret.encode(), qs.encode(), hashlib.sha256).hexdigest()

    def get_prices(self):
        try:
            r = requests.get(self.BASE+'/api/v3/ticker/price', timeout=10)
            return {i['symbol']: float(i['price']) for i in r.json()}
        except Exception as e:
            print(f'Error precios: {e}'); return {}

    def load(self, min_usd=1):
        if not self.api_key:
            print('Sin API keys — configura BINANCE_API_KEY y BINANCE_API_SECRET')
            return False
        ts  = int(time.time() * 1000)
        sig = self._sign({'timestamp': ts})
        hdrs = {'X-MBX-APIKEY': self.api_key}
        try:
            r = requests.get(self.BASE+'/api/v3/account', headers=hdrs,
                             params={'timestamp': ts, 'signature': sig}, timeout=10)
            r.raise_for_status()
            acc = r.json()
        except Exception as e:
            print(f'Error cuenta: {e}'); return False
        prices = self.get_prices()
        self.assets = []
        for b in acc.get('balances', []):
            total = float(b['free']) + float(b['locked'])
            if total <= 0: continue
            sym = b['asset']
            cp  = 1.0 if sym in ('USDT','BUSD','USDC','TUSD') else prices.get(sym+'USDT', 0)
            val = total * cp
            if val >= min_usd:
                self.assets.append({'symbol': sym, 'qty': total, 'free': float(b['free']),
                                    'locked': float(b['locked']), 'price': cp,
                                    'value': val, 'buy_price': 0})
        self.assets.sort(key=lambda x: x['value'], reverse=True)
        tv = sum(a['value'] for a in self.assets)
        print(f'{len(self.assets)} activos  |  Valor total: ${tv:,.2f}')
        return True

    def set_buy_price(self, symbol, buy_price):
        for a in self.assets:
            if a['symbol'] == symbol.upper():
                a['buy_price'] = float(buy_price)
                print(f'Precio compra {symbol}: ${buy_price}')
                return
        print(f'No encontrado: {symbol}')

    def as_dataframe(self):
        rows = []
        for a in self.assets:
            cost = a['qty'] * a['buy_price'] if a['buy_price'] else 0
            pl   = a['value'] - cost if a['buy_price'] else None
            plp  = pl / cost * 100   if cost > 0      else None
            rows.append({
                'Activo'       : a['symbol'],
                'Cantidad'     : round(a['qty'], 8),
                'Precio Compra': round(a['buy_price'], 2) if a['buy_price'] else None,
                'Precio Actual': round(a['price'], 6),
                'Valor USD'    : round(a['value'], 2),
                'G/P ($)'      : round(pl, 2)  if pl  is not None else None,
                'G/P (%)'      : round(plp, 2) if plp is not None else None,
            })
        return pd.DataFrame(rows).set_index('Activo')

In [ ]:
API_KEY    = os.environ.get('BINANCE_API_KEY', '')
API_SECRET = os.environ.get('BINANCE_API_SECRET', '')

portfolio = BinancePortfolio(API_KEY, API_SECRET)
pf_ok = portfolio.load(min_usd=1)

if pf_ok:
    # Ajusta tus precios de compra aqui
    portfolio.set_buy_price('BTC', 45000)
    portfolio.set_buy_price('ETH', 2500)
    portfolio.set_buy_price('BNB', 300)
    display(portfolio.as_dataframe())

## 💾 6. Reporte HTML Completo

Genera `crypto_analysis.html` auto-contenido con todo el análisis.

In [ ]:
senal_divs = ''.join(
    '<div style="padding:8px;margin:4px 0;background:#252525;border-left:3px solid #667eea;border-radius:4px">'
    + str(i+1) + '. ' + s + '</div>'
    for i, s in enumerate(senales)
)

if tp:
    oco_rows = (
        '<tr><td>Precio actual</td><td><b>$' + f'{precio:,.2f}' + '</b></td></tr>'
        '<tr><td>TP Limit</td><td style="color:#00c853"><b>$' + f'{tp:,.2f}' + '</b></td></tr>'
        '<tr><td>Activador SL</td><td style="color:#ff9100"><b>$' + f'{act_sl:,.2f}' + '</b></td></tr>'
        '<tr><td>SL Limit</td><td style="color:#ff1744"><b>$' + f'{sl:,.2f}' + '</b></td></tr>'
        '<tr><td>Risk/Reward</td><td>1:' + f'{rr:.2f}' + '</td></tr>'
    )
else:
    oco_rows = '<tr><td colspan="2" style="color:#ff9100">No operar</td></tr>'

ind_rows = (
    '<tr><td>EMA 4/20/50/200</td><td>$' + f'{ema_4[-1]:,.2f}' + ' / $' + f'{ema_20[-1]:,.2f}'
    + ' / $' + f'{ema_50[-1]:,.2f}' + ' / $' + f'{ema_200[-1]:,.2f}' + '</td></tr>'
    '<tr><td>RSI 14 / RSI 5</td><td>' + f'{rsi[-1]:.2f}' + ' / ' + f'{rsi_5[-1]:.2f}' + '</td></tr>'
    '<tr><td>MACD / Signal</td><td>' + f'{macd_v[-1]:.4f}' + ' / ' + f'{signal_v[-1]:.4f}' + '</td></tr>'
    '<tr><td>ADX (+DI / -DI)</td><td>' + f'{adx[-1]:.2f}' + ' (' + f'{plus_di[-1]:.2f}' + ' / ' + f'{minus_di[-1]:.2f}' + ')</td></tr>'
    '<tr><td>ATR</td><td>$' + f'{atr[-1]:,.2f}' + '</td></tr>'
    '<tr><td>Estoc. %K / %D</td><td>' + f'{slowk[-1]:.2f}' + ' / ' + f'{slowd[-1]:.2f}' + '</td></tr>'
    '<tr><td>MFI</td><td>' + f'{mfi[-1]:.2f}' + '</td></tr>'
    '<tr><td>Bollinger Sup/Inf</td><td>$' + f'{upper[-1]:,.2f}' + ' / $' + f'{lower_bb[-1]:,.2f}' + '</td></tr>'
)

pf_section = '<p style="color:#888">Portfolio no disponible (configura API keys)</p>'
if pf_ok and portfolio.assets:
    tv  = sum(a['value'] for a in portfolio.assets)
    tpl = sum(a['value'] - a['qty']*a['buy_price'] for a in portfolio.assets if a['buy_price'])
    pl_color = '#00c853' if tpl >= 0 else '#ff1744'
    pf_rows = ''
    for a in portfolio.assets:
        cost = a['qty']*a['buy_price'] if a['buy_price'] else 0
        pl   = a['value'] - cost if a['buy_price'] else 0
        plp  = pl/cost*100 if cost > 0 else 0
        cl   = '#00c853' if pl >= 0 else '#ff1744'
        bp   = ('$'+f'{a["buy_price"]:,.2f}') if a['buy_price'] else 'N/A'
        pf_rows += (
            '<tr><td><b>' + a['symbol'] + '</b></td>'
            '<td>' + f'{a["qty"]:.6f}' + '</td>'
            '<td>' + bp + '</td>'
            '<td>$' + f'{a["price"]:.4f}' + '</td>'
            '<td><b>$' + f'{a["value"]:,.2f}' + '</b></td>'
            '<td style="color:' + cl + '">' + f'${pl:+,.2f}' + '</td>'
            '<td style="color:' + cl + '">' + f'{plp:+.2f}%' + '</td></tr>'
        )
    pf_section = (
        '<p style="margin-bottom:12px">Valor total: <b>$' + f'{tv:,.2f}' + '</b>'
        ' | G/P: <b style="color:' + pl_color + '">$' + f'{tpl:+,.2f}' + '</b></p>'
        '<table><tr><th>Activo</th><th>Cantidad</th><th>Compra</th>'
        '<th>Actual</th><th>Valor</th><th>G/P$</th><th>G/P%</th></tr>'
        + pf_rows + '</table>'
    )

ts_str = datetime.now().strftime('%d/%m/%Y %H:%M:%S')
d1_str = data.index[-1].strftime('%d/%m/%Y %H:%M')
chart_html = fig.to_html(include_plotlyjs='cdn', full_html=False)

CSS = (
    '* {margin:0;padding:0;box-sizing:border-box}'
    'body {font-family:"Segoe UI",sans-serif;background:#0a0a0a;color:#eee;padding:24px}'
    '.wrap {max-width:1600px;margin:0 auto}'
    '.card {background:#141414;border-radius:12px;padding:24px;margin-bottom:20px}'
    'h2 {color:#667eea;font-size:1.15em;margin-bottom:14px}'
    'table {width:100%;border-collapse:collapse}'
    'th {background:#667eea;color:#fff;padding:10px;text-align:left}'
    'td {padding:10px;border-bottom:1px solid #222}'
    'tr:hover td {background:#1a1a1a}'
)

html = (
    '<!DOCTYPE html><html lang="es"><head>'
    '<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">'
    '<title>Crypto Analysis | ' + timeframe + '</title>'
    '<style>' + CSS + '</style></head><body><div class="wrap">'

    '<div class="card" style="text-align:center;background:linear-gradient(135deg,#1a1a2e,#16213e)">'
    '<h1 style="color:#fff;font-size:1.8em;margin-bottom:10px">CRYPTO ANALYSIS — ' + TICKER + '</h1>'
    '<div style="font-size:2.4em;font-weight:bold;margin:12px 0">$' + f'{precio:,.2f}' + '</div>'
    '<div style="opacity:.8">Timeframe: ' + timeframe + ' | ' + tendencia + ' | ' + d1_str + '</div>'
    '</div>'

    '<div class="card" style="border:2px solid ' + c + '">'
    '<h2>Recomendacion: <span style="color:' + c + '">' + rec + '</span>'
    ' &nbsp; <span style="font-size:0.85em;color:#aaa">' + f'{puntos:+d}' + '/20</span></h2>'
    '<table><tr><th>Parametro</th><th>Valor</th></tr>' + oco_rows + '</table>'
    '</div>'

    '<div class="card"><h2>Senales (' + str(len(senales)) + ')</h2>' + senal_divs + '</div>'

    '<div class="card"><h2>Indicadores</h2>'
    '<table><tr><th>Indicador</th><th>Valor</th></tr>' + ind_rows + '</table></div>'

    '<div class="card"><h2>Portfolio Binance</h2>' + pf_section + '</div>'

    '<div class="card"><h2>Grafico Interactivo</h2>' + chart_html + '</div>'

    '<div style="text-align:center;color:#444;padding:16px">Generado el ' + ts_str + '</div>'
    '</div></body></html>'
)

output_file = 'crypto_analysis.html'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(html)
print(f'Reporte guardado: {os.path.abspath(output_file)}')
print(f'Tamanio: {os.path.getsize(output_file):,} bytes')

In [ ]:
import webbrowser
try:
    webbrowser.open('file://' + os.path.abspath(output_file))
    print('Abriendo en el navegador...')
except Exception as e:
    print('Abre manualmente: ' + output_file)

---

## 📈 7. Análisis de ETFs

Mismo sistema de scoring que el análisis de BTC, aplicado a una lista de ETFs configurables.

| ETF | Descripción |
|---|---|
| SPY | S&P 500 |
| QQQ | Nasdaq 100 |
| GLD | Oro |
| IWM | Russell 2000 (Small-caps) |
| TLT | Bonos del Tesoro 20+ años |
| XLF | Sector Financiero |
| XLE | Sector Energía |
| ARKK | ARK Innovation |
| EEM | Mercados Emergentes |
| VTI | Mercado Total EE.UU. |

> Edita `ETF_LIST` y `ETF_CHOICE` según tus intereses.

In [ ]:
# Tickers a analizar — agrega o quita según necesites
ETF_LIST = {
    'SPY' : 'S&P 500',
    'QQQ' : 'Nasdaq 100',
    'GLD' : 'Oro',
    'IWM' : 'Russell 2000',
    'TLT' : 'Bonos 20+ años',
    'XLF' : 'Sector Financiero',
    'XLE' : 'Sector Energia',
}

# Timeframe para ETFs (mismas opciones que crypto)
# 1=1d 2=12h 3=6h 4=4h 5=1h 6=30m 7=15m 8=5m
ETF_CHOICE = '1'

ETF_TIMEFRAMES = {
    '1': ('1d',  '2y',  '1 DIA'),
    '2': ('1h',  '60d', '12 HORAS'),
    '3': ('1h',  '60d', '6 HORAS'),
    '4': ('1h',  '60d', '4 HORAS'),
    '5': ('1h',  '7d',  '1 HORA'),
    '6': ('30m', '7d',  '30 MIN'),
    '7': ('15m', '5d',  '15 MIN'),
    '8': ('5m',  '5d',  '5 MIN'),
}

etf_interval, etf_period, etf_timeframe = ETF_TIMEFRAMES[ETF_CHOICE]
print(f'Timeframe ETF: {etf_timeframe}  ({etf_interval} / {etf_period})')

### Función de análisis

In [ ]:
def analyze_asset(ticker, nombre, interval='1d', period='2y'):
    raw = yf.download(ticker, period=period, interval=interval, progress=False)
    if raw.empty:
        print(f'Sin datos: {ticker}'); return None
    data = raw.copy()
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.droplevel(1)

    close  = np.asarray(data['Close'].dropna().values,  dtype=float).flatten()
    high   = np.asarray(data['High'].dropna().values,   dtype=float).flatten()
    low    = np.asarray(data['Low'].dropna().values,    dtype=float).flatten()
    volume = np.asarray(data['Volume'].dropna().values, dtype=float).flatten()
    open_p = np.asarray(data['Open'].dropna().values,   dtype=float).flatten()

    if len(close) < 30:
        print(f'Datos insuficientes: {ticker}'); return None

    ema_4    = talib.EMA(close, timeperiod=4)
    ema_20   = talib.EMA(close, timeperiod=20)
    ema_25   = talib.EMA(close, timeperiod=25)
    ema_50   = talib.EMA(close, timeperiod=50)
    ema_200  = talib.EMA(close, timeperiod=200)
    rsi_v    = talib.RSI(close, timeperiod=14)
    rsi5_v   = talib.RSI(close, timeperiod=5)
    macd_v, sig_v, hist_v = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
    sk, sd   = talib.STOCH(high, low, close, fastk_period=14, slowk_period=3, slowd_period=3)
    adx_v    = talib.ADX(high, low, close, timeperiod=14)
    pdi_v    = talib.PLUS_DI(high, low, close, timeperiod=14)
    mdi_v    = talib.MINUS_DI(high, low, close, timeperiod=14)
    atr_v    = talib.ATR(high, low, close, timeperiod=14)
    obv_v    = talib.OBV(close, volume)
    obv_ema  = talib.EMA(obv_v, timeperiod=20)
    up, mid, dn = talib.BBANDS(close, timeperiod=20, nbdevup=2, nbdevdn=2)
    mfi_v    = talib.MFI(high, low, close, volume, timeperiod=14)

    precio  = float(close[-1])
    atr_val = float(atr_v[-1]) if not np.isnan(atr_v[-1]) else 0
    bb_w    = float((up[-1]-dn[-1])/mid[-1]*100) if not np.isnan(up[-1]) else 0

    puntos = 0
    senales = []

    # EMAs
    if not any(np.isnan([ema_20[-1], ema_50[-1], ema_200[-1]])):
        if ema_20[-1] > ema_50[-1] > ema_200[-1]:
            puntos += 3; tendencia = 'ALCISTA'; senales.append('✅ EMAs alcistas (20>50>200)')
        elif ema_20[-1] < ema_50[-1] < ema_200[-1]:
            puntos -= 3; tendencia = 'BAJISTA'; senales.append('❌ EMAs bajistas (20<50<200)')
        else:
            tendencia = 'NEUTRAL'; senales.append('⚠️ EMAs sin alineacion')
    else:
        tendencia = 'NEUTRAL'; senales.append('⚠️ EMAs sin datos')

    if not np.isnan(ema_4[-1]):
        if precio > ema_4[-1]: puntos += 1; senales.append(f'✅ Precio > EMA4 (${ema_4[-1]:,.2f})')
        else:                  puntos -= 1; senales.append(f'❌ Precio < EMA4 (${ema_4[-1]:,.2f})')

    if not np.isnan(rsi_v[-1]):
        if   rsi_v[-1] < 30: puntos += 2; senales.append(f'✅ RSI sobreventa ({rsi_v[-1]:.1f})')
        elif rsi_v[-1] > 70: puntos -= 2; senales.append(f'❌ RSI sobrecompra ({rsi_v[-1]:.1f})')
        elif rsi_v[-1] > 50: puntos += 1; senales.append(f'✅ RSI alcista ({rsi_v[-1]:.1f})')
        else:                 puntos -= 1; senales.append(f'❌ RSI bajista ({rsi_v[-1]:.1f})')

    if not np.isnan(rsi5_v[-1]):
        if   rsi5_v[-1] < 20: puntos += 1; senales.append(f'✅ RSI5 sobreventa ({rsi5_v[-1]:.1f})')
        elif rsi5_v[-1] > 80: puntos -= 1; senales.append(f'❌ RSI5 sobrecompra ({rsi5_v[-1]:.1f})')

    if not any(np.isnan([macd_v[-1], sig_v[-1]])):
        if macd_v[-1] > sig_v[-1]: puntos += 2; senales.append('✅ MACD > Signal')
        else:                       puntos -= 2; senales.append('❌ MACD < Signal')
        if not any(np.isnan([hist_v[-1], hist_v[-2], hist_v[-3]])):
            if   hist_v[-1] > hist_v[-2] > hist_v[-3]: puntos += 1; senales.append('✅ Histograma creciendo')
            elif hist_v[-1] < hist_v[-2] < hist_v[-3]: puntos -= 1; senales.append('❌ Histograma decreciendo')

    if not np.isnan(adx_v[-1]):
        if adx_v[-1] > 25:
            if pdi_v[-1] > mdi_v[-1]: puntos += 2; senales.append(f'✅ Tendencia fuerte alcista (ADX {adx_v[-1]:.1f})')
            else:                      puntos -= 2; senales.append(f'❌ Tendencia fuerte bajista (ADX {adx_v[-1]:.1f})')
        else: senales.append(f'⚠️ ADX debil ({adx_v[-1]:.1f})')

    if not np.isnan(sk[-1]):
        if sk[-1] < 20 and sd[-1] < 20:   puntos += 1; senales.append('✅ Estocastico sobreventa')
        elif sk[-1] > 80 and sd[-1] > 80:  puntos -= 1; senales.append('❌ Estocastico sobrecompra')
        if not any(np.isnan([sk[-2], sd[-2]])):
            if   sk[-1] > sd[-1] and sk[-2] < sd[-2]: puntos += 1; senales.append('✅ Cruce alcista estocastico')
            elif sk[-1] < sd[-1] and sk[-2] > sd[-2]: puntos -= 1; senales.append('❌ Cruce bajista estocastico')

    if not np.isnan(mfi_v[-1]):
        if   mfi_v[-1] < 20: puntos += 1; senales.append(f'✅ MFI sobreventa ({mfi_v[-1]:.1f})')
        elif mfi_v[-1] > 80: puntos -= 1; senales.append(f'❌ MFI sobrecompra ({mfi_v[-1]:.1f})')

    if not any(np.isnan([obv_v[-1], obv_ema[-1]])):
        if obv_v[-1] > obv_ema[-1]: puntos += 1; senales.append('✅ OBV > EMA (presion compradora)')
        else:                        puntos -= 1; senales.append('❌ OBV < EMA (presion vendedora)')

    if not np.isnan(dn[-1]):
        if   precio < dn[-1]:  puntos += 1; senales.append('✅ Precio bajo banda Bollinger')
        elif precio > up[-1]:  puntos -= 1; senales.append('❌ Precio sobre banda Bollinger')
        if bb_w < 2: senales.append(f'⚠️ Bollinger comprimidas ({bb_w:.2f}%)')

    if   puntos >= 8:  rec='COMPRA FUERTE'; c='#00c853'; tp=precio+atr_val*3;   sl=precio-atr_val*1.5
    elif puntos >= 4:  rec='COMPRA';         c='#64dd17'; tp=precio+atr_val*2.5; sl=precio-atr_val*1.5
    elif puntos >= 1:  rec='COMPRA DEBIL';   c='#ffd600'; tp=precio+atr_val*2;   sl=precio-atr_val*1.2
    elif puntos <= -8: rec='VENTA FUERTE';   c='#ff1744'; tp=precio-atr_val*3;   sl=precio+atr_val*1.5
    elif puntos <= -4: rec='VENTA';          c='#ff5252'; tp=precio-atr_val*2.5; sl=precio+atr_val*1.5
    elif puntos <= -1: rec='VENTA DEBIL';    c='#ff9100'; tp=precio-atr_val*2;   sl=precio+atr_val*1.2
    else:              rec='NEUTRAL';         c='#9e9e9e'; tp=sl=None

    rr = abs((tp-precio)/(precio-sl)) if tp and sl and (precio-sl)!=0 else 0

    # Grafico
    fig = make_subplots(
        rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.04,
        subplot_titles=(ticker+' Precio+EMAs+BB', 'Volumen', 'RSI', 'MACD', 'Estocastico'),
        row_heights=[0.40, 0.10, 0.15, 0.15, 0.15]
    )
    fig.add_trace(go.Candlestick(x=data.index, open=open_p, high=high, low=low, close=close, name=ticker), row=1, col=1)
    fig.add_trace(go.Scatter(x=data.index, y=ema_4,   mode='lines', name='EMA4',   line=dict(color='#ab47bc', width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=data.index, y=ema_20,  mode='lines', name='EMA20',  line=dict(color='#fdd835', width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=data.index, y=ema_50,  mode='lines', name='EMA50',  line=dict(color='#ef5350', width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=data.index, y=ema_200, mode='lines', name='EMA200', line=dict(color='#fff',    width=1, dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=data.index, y=up, mode='lines', name='BB Sup', line=dict(color='#78909c', width=1, dash='dot')), row=1, col=1)
    fig.add_trace(go.Scatter(x=data.index, y=dn, mode='lines', name='BB Inf',
                             line=dict(color='#78909c', width=1, dash='dot'),
                             fill='tonexty', fillcolor='rgba(120,144,156,0.06)'), row=1, col=1)
    fig.add_trace(go.Bar(x=data.index, y=volume/1e6, name='Vol(M)', marker_color='#4472c4'), row=2, col=1)
    fig.add_trace(go.Scatter(x=data.index, y=rsi_v,  mode='lines', name='RSI14', line=dict(color='#ab47bc')), row=3, col=1)
    fig.add_trace(go.Scatter(x=data.index, y=rsi5_v, mode='lines', name='RSI5',  line=dict(color='#00e5ff', width=1)), row=3, col=1)
    fig.add_hline(y=70, row=3, col=1, line_dash='dash', line_color='#ef5350', line_width=1)
    fig.add_hline(y=30, row=3, col=1, line_dash='dash', line_color='#66bb6a', line_width=1)
    fig.add_trace(go.Scatter(x=data.index, y=macd_v, mode='lines', name='MACD',   line=dict(color='#42a5f5')), row=4, col=1)
    fig.add_trace(go.Scatter(x=data.index, y=sig_v,  mode='lines', name='Signal', line=dict(color='#ef5350')), row=4, col=1)
    hcol = ['#00c853' if h > 0 else '#ff1744' for h in hist_v]
    fig.add_trace(go.Bar(x=data.index, y=hist_v, name='Hist', marker_color=hcol), row=4, col=1)
    fig.add_trace(go.Scatter(x=data.index, y=sk, mode='lines', name='%K', line=dict(color='#42a5f5')), row=5, col=1)
    fig.add_trace(go.Scatter(x=data.index, y=sd, mode='lines', name='%D', line=dict(color='#ef5350')), row=5, col=1)
    fig.add_hline(y=80, row=5, col=1, line_dash='dash', line_color='#ef5350', line_width=1)
    fig.add_hline(y=20, row=5, col=1, line_dash='dash', line_color='#66bb6a', line_width=1)
    fig.update_layout(
        title=dict(text=ticker+' | '+nombre+' | '+rec+' | Score: '+f'{puntos:+d}'+'/20', font=dict(size=14)),
        height=1100, showlegend=True, template='plotly_dark',
        xaxis_rangeslider_visible=False,
        legend=dict(orientation='h', y=1.01, x=0, font=dict(size=9))
    )

    return {
        'ticker': ticker, 'nombre': nombre, 'precio': precio,
        'puntos': puntos, 'rec': rec, 'tendencia': tendencia, 'color': c,
        'rsi': float(rsi_v[-1]) if not np.isnan(rsi_v[-1]) else 0,
        'adx': float(adx_v[-1]) if not np.isnan(adx_v[-1]) else 0,
        'rsi5': float(rsi5_v[-1]) if not np.isnan(rsi5_v[-1]) else 0,
        'macd': float(macd_v[-1]) if not np.isnan(macd_v[-1]) else 0,
        'atr': atr_val, 'tp': tp, 'sl': sl, 'rr': rr,
        'senales': senales, 'fig': fig, 'data': data,
        'ema_20': float(ema_20[-1]), 'ema_50': float(ema_50[-1]),
        'ema_200': float(ema_200[-1]),
    }

print('Funcion analyze_asset() lista')

### Descarga y análisis de cada ETF

In [ ]:
resultados = {}
for ticker, nombre in ETF_LIST.items():
    print(f'  {ticker:6s} — {nombre}...', end=' ')
    r = analyze_asset(ticker, nombre, etf_interval, etf_period)
    if r:
        resultados[ticker] = r
        print(f'{r["rec"]:15s} | Score: {r["puntos"]:+3d} | RSI: {r["rsi"]:.1f}')
    else:
        print('Error')

print(f'Analizados: {len(resultados)}/{len(ETF_LIST)}')

### Tabla comparativa

In [ ]:
filas = []
for r in resultados.values():
    filas.append({
        'ETF'          : r['ticker'],
        'Nombre'       : r['nombre'],
        'Precio'       : f'${r["precio"]:,.2f}',
        'Score'        : f'{r["puntos"]:+d}/20',
        'Recomendacion': r['rec'],
        'Tendencia'    : r['tendencia'],
        'RSI 14'       : f'{r["rsi"]:.1f}',
        'ADX'          : f'{r["adx"]:.1f}',
        'EMA20/50/200' : f'{r["ema_20"]:,.2f} / {r["ema_50"]:,.2f} / {r["ema_200"]:,.2f}',
        'TP'           : f'${r["tp"]:,.2f}' if r['tp'] else 'N/A',
        'SL'           : f'${r["sl"]:,.2f}' if r['sl'] else 'N/A',
        'R:R'          : f'1:{r["rr"]:.2f}' if r['rr'] else 'N/A',
    })

df_comp = pd.DataFrame(filas).set_index('ETF')

# Colores por recomendacion
def color_rec(val):
    if 'COMPRA FUERTE' in val: return 'background-color:#1a3a1a;color:#00c853;font-weight:bold'
    if 'COMPRA DEBIL'  in val: return 'background-color:#2a2a00;color:#ffd600'
    if 'COMPRA'        in val: return 'background-color:#1a2a1a;color:#64dd17;font-weight:bold'
    if 'VENTA FUERTE'  in val: return 'background-color:#3a1a1a;color:#ff1744;font-weight:bold'
    if 'VENTA DEBIL'   in val: return 'background-color:#2a1a00;color:#ff9100'
    if 'VENTA'         in val: return 'background-color:#2a1a1a;color:#ff5252;font-weight:bold'
    return 'color:#9e9e9e'

def color_score(val):
    try:
        n = int(val.split('/')[0])
        if n >= 4:  return 'color:#00c853;font-weight:bold'
        if n <= -4: return 'color:#ff1744;font-weight:bold'
        return 'color:#ffd600'
    except: return ''

display(
    df_comp.style
    .map(color_rec,   subset=['Recomendacion'])
    .map(color_score, subset=['Score'])
)

### Ranking por puntuación

In [ ]:
ranking = sorted(resultados.values(), key=lambda x: x['puntos'], reverse=True)

bar_tickers = [r['ticker'] for r in ranking]
bar_scores  = [r['puntos'] for r in ranking]
bar_colors  = [r['color']  for r in ranking]
bar_names   = [r['nombre'] for r in ranking]

fig_rank = go.Figure(go.Bar(
    x=bar_tickers,
    y=bar_scores,
    marker_color=bar_colors,
    text=[f'{s:+d}' for s in bar_scores],
    textposition='outside',
    customdata=bar_names,
    hovertemplate='<b>%{x}</b><br>%{customdata}<br>Score: %{y:+d}/20<extra></extra>'
))
fig_rank.update_layout(
    title='Ranking ETFs por Score (+/-20)',
    template='plotly_dark', height=420,
    yaxis=dict(title='Score', zeroline=True, zerolinecolor='#555', range=[-22, 22]),
    xaxis_title='ETF',
    showlegend=False
)
fig_rank.add_hline(y=4,  line_dash='dash', line_color='#64dd17', line_width=1, annotation_text='Compra')
fig_rank.add_hline(y=-4, line_dash='dash', line_color='#ff5252', line_width=1, annotation_text='Venta')
fig_rank.show()

### Gráficos individuales por ETF

In [ ]:
for ticker, r in resultados.items():
    header = (
        '<div style="font-family:monospace;background:#141414;padding:12px 20px;'
        'border-radius:8px;margin:16px 0 4px;border-left:4px solid ' + r['color'] + '">'
        '<b style="font-size:1.3em">' + ticker + '</b>'
        ' &nbsp; <span style="color:#aaa">' + r['nombre'] + '</span>'
        ' &nbsp;|&nbsp; <span style="color:' + r['color'] + ';font-weight:bold">' + r['rec'] + '</span>'
        ' &nbsp; <span style="color:#888">' + f'{r["puntos"]:+d}' + '/20</span>'
        ' &nbsp;|&nbsp; RSI: <b>' + f'{r["rsi"]:.1f}' + '</b>'
        ' | ADX: <b>' + f'{r["adx"]:.1f}' + '</b>'
        ' | Precio: <b>$' + f'{r["precio"]:,.2f}' + '</b>'
        '</div>'
    )
    display(HTML(header))
    r['fig'].show()

### Señales detalladas por ETF

In [ ]:
for ticker, r in resultados.items():
    senal_html = ''.join(
        '<div style="padding:6px 10px;margin:3px 0;background:#1a1a1a;'
        'border-left:3px solid #667eea;border-radius:4px;font-size:13px">'
        + str(j+1) + '. ' + s + '</div>'
        for j, s in enumerate(r['senales'])
    )
    block = (
        '<div style="font-family:monospace;background:#111;padding:16px;'
        'border-radius:10px;margin:10px 0;max-width:800px">'
        '<h3 style="color:' + r['color'] + ';margin:0 0 10px">'
        + ticker + ' — ' + r['nombre'] + '</h3>'
        + senal_html +
        '</div>'
    )
    display(HTML(block))

### Reporte HTML de ETFs

In [ ]:
CSS_ETF = (
    '* {margin:0;padding:0;box-sizing:border-box}'
    'body {font-family:"Segoe UI",sans-serif;background:#0a0a0a;color:#eee;padding:24px}'
    '.wrap {max-width:1600px;margin:0 auto}'
    '.card {background:#141414;border-radius:12px;padding:20px;margin-bottom:16px}'
    'h1 {color:#fff;text-align:center;margin-bottom:8px}'
    'h2 {color:#667eea;font-size:1.1em;margin-bottom:12px}'
    'h3 {margin:0 0 10px}'
    'table {width:100%;border-collapse:collapse}'
    'th {background:#1e1e5e;color:#aaa;padding:8px;text-align:left;font-size:12px}'
    'td {padding:8px;border-bottom:1px solid #222;font-size:13px}'
    'tr:hover td {background:#1a1a1a}'
    '.grid {display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:14px}'
    '.signal {padding:6px 10px;margin:3px 0;background:#1a1a1a;'
    'border-left:3px solid #667eea;border-radius:4px;font-size:12px}'
)

rows_etf = ''
for r in resultados.values():
    rows_etf += (
        '<tr>'
        '<td><b>' + r['ticker'] + '</b></td>'
        '<td>' + r['nombre'] + '</td>'
        '<td>$' + f'{r["precio"]:,.2f}' + '</td>'
        '<td style="color:' + r['color'] + ';font-weight:bold">' + rec + '</td>'
        '<td style="color:' + r['color'] + '">' + f'{r["puntos"]:+d}' + '/20</td>'
        '<td>' + r['tendencia'] + '</td>'
        '<td>' + f'{r["rsi"]:.1f}' + '</td>'
        '<td>' + f'{r["adx"]:.1f}' + '</td>'
        '<td>' + (f'${r["tp"]:,.2f}' if r["tp"] else 'N/A') + '</td>'
        '<td>' + (f'${r["sl"]:,.2f}' if r["sl"] else 'N/A') + '</td>'
        '<td>' + (f'1:{r["rr"]:.2f}'  if r["rr"] else 'N/A') + '</td>'
        '</tr>'
    )

cards_etf = ''
for r in resultados.values():
    sigs = ''.join('<div class="signal">' + str(j+1) + '. ' + s + '</div>' for j, s in enumerate(r['senales']))
    cards_etf += (
        '<div class="card" style="border-left:3px solid ' + r['color'] + '">'
        '<h3 style="color:' + r['color'] + '">' + r['ticker'] + ' — ' + r['nombre'] + '</h3>'
        '<p style="margin-bottom:10px">Score: <b>' + f'{r["puntos"]:+d}' + '/20</b>'
        ' | RSI: ' + f'{r["rsi"]:.1f}' + ' | ADX: ' + f'{r["adx"]:.1f}' + '</p>'
        + sigs + '</div>'
    )

charts_etf = ''.join(
    '<div class="card"><h2>' + r['ticker'] + ' | ' + r['nombre'] + '</h2>'
    + r['fig'].to_html(include_plotlyjs='cdn' if j == 0 else False, full_html=False)
    + '</div>'
    for j, r in enumerate(resultados.values())
)

ts_etf = datetime.now().strftime('%d/%m/%Y %H:%M:%S')

html_etf = (
    '<!DOCTYPE html><html lang="es"><head>'
    '<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">'
    '<title>ETF Analysis | ' + etf_timeframe + '</title>'
    '<style>' + CSS_ETF + '</style></head><body><div class="wrap">'
    '<div class="card" style="text-align:center;background:linear-gradient(135deg,#0d1b2a,#1b2838)">'
    '<h1>ETF ANALYSIS</h1>'
    '<div style="opacity:.7">Timeframe: ' + etf_timeframe + ' | ' + str(len(resultados)) + ' ETFs analizados</div>'
    '</div>'
    '<div class="card"><h2>Resumen comparativo</h2>'
    '<table><tr><th>ETF</th><th>Nombre</th><th>Precio</th><th>Rec.</th>'
    '<th>Score</th><th>Tendencia</th><th>RSI</th><th>ADX</th>'
    '<th>TP</th><th>SL</th><th>R:R</th></tr>'
    + rows_etf + '</table></div>'
    '<div class="card"><h2>Señales por ETF</h2><div class="grid">' + cards_etf + '</div></div>'
    + charts_etf +
    '<div style="text-align:center;color:#444;padding:16px">Generado el ' + ts_etf + '</div>'
    '</div></body></html>'
)

etf_file = 'etf_analysis.html'
with open(etf_file, 'w', encoding='utf-8') as f:
    f.write(html_etf)
print(f'Reporte ETF guardado: {etf_file}  ({os.path.getsize(etf_file):,} bytes}')

In [ ]:
import webbrowser
try:
    webbrowser.open('file://' + os.path.abspath(etf_file))
    print('Abriendo reporte ETF en navegador...')
except:
    print('Abre manualmente: ' + etf_file)